https://towardsdatascience.com/pandas-apply-map-or-transform-dd931659e9cf/

In [2]:
# Importamos las librerías necesarias
from zipfile import ZipFile
from pathlib import Path
import pandas as pd
import numpy as np

try:
    datos = Path.cwd() / "datos.zip" # Cargamos el fichero .zip desde el directorio actual.

    with ZipFile(datos) as z: # Creamos los dataframes en base a los ficheros del .zip con sus parámetros correspondientes
        with z.open("datNotas.csv") as f: df_notas = pd.read_csv(f, encoding="latin-1", sep=";")
        with z.open("datMatriculas.csv") as f: df_matriculas = pd.read_csv(f, encoding="latin-1", sep=";")
        with z.open("datFaltas.csv") as f: df_faltas = pd.read_csv(f, encoding="latin-1", sep=";")
        with z.open("datMaterias.csv") as f: df_materias = pd.read_csv(f, encoding="latin-1", sep=";")
        with z.open("datUnidades.csv") as f: df_unidades = pd.read_csv(f, encoding="latin-1", sep=";")
        

except FileNotFoundError as e: print(f"Se ha producido un error: {e}")
except UnicodeEncodeError as e: print(f"Se ha producido un error: {e}")

# Cargamos solo las columnas necesarias
df_notas = df_notas[["MATRICULA", "ANNO", "CURSO", "EVALUACION", "MATERIA", "CL_MATERIA", "NOTA"]]
df_matriculas = df_matriculas[["MATRICULA", "ETAPA", "ANNO", "ESTUDIOS", "GRUPO", "ESTADOMATRICULA"]]
df_faltas = df_faltas[["EXPEDIENTE", "ANNO", "FECHA_FALTA", "ESTUDIOS", "GRUPO", "MATERIA", "AUSENCIAS", "RETRASOS", "JUSTIFICADAS"]]
df_materias = df_materias[["CODIGO", "DESCRIPCION", "ABREVIATURA"]]
df_unidades = df_unidades[["ANNO", "GRUPO", "ESTUDIO", "CURSO"]] # Opcional. Falta "TUTOR"



In [3]:
df_notas = df_notas[df_notas["EVALUACION"].isin(["1ª Evaluación", "2ª Evaluación"])] # Filtramos a los alumnos por primera y Segunda Evaluación
display(df_notas["EVALUACION"].value_counts(dropna=True))

display(df_notas["NOTA"].value_counts(dropna=True))

EVALUACION
1ª Evaluación    18113
2ª Evaluación    14130
Name: count, dtype: int64

NOTA
Insuficiente             2572
7                        1553
8                        1449
Notable                  1342
6                        1318
Suficiente               1152
5                        1087
1                        1060
4                        1007
9                         956
Sobresaliente             912
Bien                      874
3                         664
2                         497
10                        408
Apto                      119
0                          22
No evaluado                10
No Apto                     7
Insuficiente-1              3
Renuncia Convocatoria       2
Name: count, dtype: int64

In [ ]:
df_notas_matriculas = pd.merge(df_notas, df_matriculas, on="MATRICULA", how="left")


lista_materias = df_notas_matriculas["MATERIA"].unique().tolist()
lista_evaluaciones = df_notas_matriculas["EVALUACION"].unique().tolist()
lista_grupos = df_notas_matriculas["GRUPO"].unique().tolist()


['2º DAM D',
 '2º DAM V',
 '2º CFGB IO',
 '2º AF V',
 '2º DAW D',
 '2º ASIR D',
 '1º CFGB SA',
 '1º ASIR D',
 'AFVmodPR',
 '2º SMR V',
 '2º SMR D',
 nan,
 'DAWmodPR',
 '2º GA V',
 '1º SMR D',
 '2º GA D',
 'SMRDmodPR',
 'SMRVmodPT',
 'GAV mod PR',
 '2ºAFD-BL',
 '1º GA V',
 'DAMVmodPR',
 '2º AF D',
 '2º AD V BL',
 '1º DAM D',
 '1º DAM V',
 'DAMVmodPT',
 '2º AD V',
 'AFVmodPT',
 '1º DAW D',
 '1º GA D',
 '2 PEFP JAR',
 'AFDmodPT',
 'ASIRmodPR',
 '2º AF V BL',
 'GAD mod PR',
 '1º SMR V',
 '1º AF D',
 '2º CFGB SA',
 'DAMDmodPR',
 'ASIR.MODUL',
 'AFDmodPR',
 'SMRDmodPT',
 'X2º AD V',
 '3ºA BL',
 '3ºB NBL',
 '2ºD NBL',
 'ADmodPT',
 '3ºC NBL',
 'SMRVmodPR',
 '4ºD DIVER',
 '2º BACH B',
 '2ºC NBL',
 '2º BACH C',
 '3ºD NBL',
 '4ºC NBL',
 '2ºB BL',
 '4ºB NBL',
 '2ºA NBL',
 '4ºA NBL',
 'ADmodPR',
 '4ºD NBL',
 '2º BACH A',
 '1ºC NBL',
 '3ºA NBL',
 '1ºA NBL',
 'X1º DAM V',
 '1º AF V',
 '4ºB BL',
 'X 4º',
 '2ºB NBL',
 '1º BACH B',
 '1ºB NBL',
 '2ºA BL',
 '4ºC BL',
 '4ºA DIVER',
 '1ºD NBL',
 '1º CFGB IO

In [5]:


# Para notas con valores como "Suficiente", "Apto", o "Sobresaliente",
# tenemos que hacer un reajuste a valores int aleatorios
notas_categoricas = {
    "Insuficiente": 4.5,
    "Insuficiente-1": 1,
    "Suficiente": 5,
    "Apto": 5,
    "Bien": 6.5,
    "Notable": 7.5,
    "Sobresaliente": 9.5,

    "No evaluado": np.nan,
    "Renuncia Convocatoria": np.nan
}

df_notas["NOTA"] = df_notas["NOTA"].replace(notas_categoricas)
df_notas["NOTA"] = pd.to_numeric(df_notas["NOTA"], errors="coerce")
#df_notas = df_notas[["NOTA"]].dropna()



display(df_notas["NOTA"].isna().sum())
display(df_notas["NOTA"].sum())

display(df_notas.describe())
display(df_notas.sort_values(df_notas["NOTA"], ascending=False))

np.int64(15248)

np.float64(98906.0)

,MATRICULA,ANNO,CL_MATERIA,NOTA
count,3.224300e+04,32243.0,32243.000000,16995.000000
mean,1.000377e+07,2025.0,63979.104054,5.819712
std,1.780956e+05,0.0,6695.340825,2.298827
min,9.743885e+06,2025.0,34022.000000,0.000000
25%,9.825457e+06,2025.0,61420.000000,4.500000
50%,1.001929e+07,2025.0,65332.000000,6.000000
75%,1.016637e+07,2025.0,66781.000000,7.500000
max,1.041092e+07,2025.0,80118.000000,10.000000


KeyError: 1         4.0
3         2.0
5        10.0
7         9.0
9         6.0
         ... 
69786     4.5
69787     NaN
69790     4.5
69791     NaN
69792     4.5
Name: NOTA, Length: 32243, dtype: float64

In [ ]:
# display(df_unidades)
# display(df_faltas)
# display(df_materias)
# display(df_matriculas)
# display(df_notas)

Python + Flask

- rendimientos:
    - por materia
    - por grupo
- absentismo

